# Single-kernel sync mutants, race-checker audit and batch invariance on a T4

In [ ]:
# Kaggle: GPU T4, internet on, private dataset rajulkabir/tileproof-src-warps attached
# (triton_equiv and tinytail source as a tarball; SOURCE.txt names the commits).
import glob, os, shutil, subprocess, sys, tarfile
os.environ["TRITON_INTERPRET"] = "0"
SRC = "/kaggle/working/tileproof_src"
tars = glob.glob("/kaggle/input/**/tileproof_src.tar.gz", recursive=True)
srcs = glob.glob("/kaggle/input/**/SOURCE.txt", recursive=True)
os.makedirs(SRC, exist_ok=True)
if tars:
    with tarfile.open(tars[0]) as t:
        t.extractall(SRC)
elif srcs:  # Kaggle unpacked the tarball
    shutil.copytree(os.path.dirname(srcs[0]), SRC, dirs_exist_ok=True)
else:
    raise SystemExit("dataset missing: " + str(glob.glob("/kaggle/input/**", recursive=True)[:50]))
print(open(SRC + "/SOURCE.txt").read())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver", "ninja", SRC + "/tinytail"], check=True)
sys.path[:0] = [SRC]
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
OUT = "/kaggle/working"

In [ ]:
import json, torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name(0))
from triton_equiv.msync import intra
r = intra.main(["--reps", "200", "--out", f"{OUT}/msync_intra.json"])

In [ ]:
from triton_equiv import batchinv
b = batchinv.main(["--gpu", "--out", f"{OUT}/batchinv.json"])
for row in b["rows"]:
    print(row["family"], row["verdict"], row["gpu"])

In [ ]:
T39 = "/kaggle/working/tri39"
p = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", T39, "triton==3.9.0"],
                   capture_output=True, text=True)
print(p.returncode, p.stdout[-500:], p.stderr[-1500:])
chk = subprocess.run([sys.executable, "-c", "import triton, triton.experimental.gsan as g; print(triton.__version__, triton.__file__)"],
                     capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=T39 + os.pathsep + SRC))
print(chk.stdout, chk.stderr[-1500:])
print(subprocess.run(["bash", "-lc", "which compute-sanitizer; ls /usr/local/cuda*/bin/compute-sanitizer; nvidia-smi --query-gpu=driver_version --format=csv"],
                     capture_output=True, text=True).stdout)

In [ ]:
from triton_equiv.msync import sanitize
s = sanitize.main(["--tools", "racecheck", "synccheck", "memcheck", "gsan", "consan", "--gsan-path", T39,
                   "--out", f"{OUT}/msync_sanitizers.json", "--log-dir", f"{OUT}/sanitizer_logs"])